# LR-ASPP stride-4 decoder

# Imports

In [ ]:
import copy
import os
import sys
import json
import time
from functools import partial

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy import ndimage as ndi
from torch import nn
from tqdm.auto import tqdm

from uavloc.data.annotated import IGNORE_LABEL
from uavloc.features.imagenet import normalise
from uavloc.features.segmentation import (CLASSES, ENCODER_STRIDE_PX, SINGLE_PASS_MAX_PX, TILE_PX, TILE_STEP_PX,
                                          LRASPPLogits, load_segmenter)
from uavloc.features.tiling import tiled_logits
from uavloc.plotting.correlation import SEG_PALETTE
from uavloc.segmentation.eval import (HOLDOUT_LOCATION, MEAN_KEY, STRIP_LOCATION, STRIP_ROWS, VAL_SETS,
                                      bilinear_argmax, evaluate_predictions, label_at_stride, load_val_sets,
                                      pad_to_multiple, results_table, run_model, upsample_logits)
from uavloc.segmentation.latency import cpu_latency, standard_tile
from uavloc.segmentation.refine.refinement_cnn import tile_starts
from uavloc.segmentation.losses import dice_loss
from uavloc.track import REPO_ROOT, set_seed

sys.path.insert(0, str(REPO_ROOT / "pocs" / "village-seg-baseline"))
import village_seg as vs

# Params

In [ ]:
ARMS = ["original", "control", "stride4"]
DECODER_WIDTH = 32
STEPS = 400
LR = 1e-4
DECODER_LR = 1e-3
WEIGHT_DECAY = 1e-4
WARMUP_FRAC = 0.05
SAT_CROP_PX = 512
SAT_BATCH = 4
RARE_CROP_P = 0.3
GAP_PX = 64
BN_BATCHES = 60
BN_SEED = 123
SEED = 42
DEVICE = "mps"
EVAL_DEVICE = "cpu"
LOG_EVERY_STEPS = 25
VAL_CROP_PX = 512
VAL_BATCH = 8
DATASET = "ua-village-annotated"
INIT_CHECKPOINT = "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt"
LATENCY_TILE = ["bushcha", "2024-06-06", 1024, 1024]
LATENCY_RUNS = 30
LATENCY_THREADS = 4
MIN_BUILDING_PX = 16
MATCH_FRAC = 0.2
EXAMPLES = [["bushcha", 0, 1664, 1472], ["birky-strip", 0, 64, 256]]
ZOOM_PX = 256
OUT_DIR = "pocs/lraspp-stride4-decoder"

In [ ]:
BUILDING = CLASSES.index("building")
STRIDE4_LAYER, LOW_LAYER, HIGH_LAYER = "3", "4", "16"
STRUCTURE = ndi.generate_binary_structure(2, 1)
STATUSES = ["separate", "merged", "missed"]
ARM_STRIDE = {"original": 8, "control": 8, "stride4": 4}

# Data

In [ ]:
set_seed(SEED)
device = torch.device(DEVICE if DEVICE != "mps" or torch.backends.mps.is_available() else "cpu")
DERIVED = REPO_ROOT / "data" / "derived"
sat = vs.load_satellite(DERIVED / DATASET, DERIVED)
vs.ban_holdouts(sat, HOLDOUT_LOCATION, STRIP_LOCATION, STRIP_ROWS, GAP_PX)
sat_train = [s for s in sat if not s.banned.all()]
sampler = vs.SatelliteSampler(sat_train, SAT_CROP_PX, RARE_CROP_P)
[(s.loc.split("-")[0], s.date) for s in sat_train]

In [ ]:
val_sets = load_val_sets(DATASET)
{name: len(images) for name, images in val_sets.items()}

# Model

In [ ]:
class Stride4LRASPP(nn.Module):
    """LR-ASPP plus one decoder step. Output = x8 upsample of its stride-8 logits (unchanged, exactly the base model's
    path) + x4 upsample of a stride-4 correction computed from the backbone's layer-3 features (24 ch) and the
    stride-8 logits upsampled x2. The correction starts at zero, so an untrained model equals the base model."""

    def __init__(self, base: LRASPPLogits, width: int) -> None:
        super().__init__()
        self.base = base
        n_classes = len(CLASSES)
        self.decoder = nn.Sequential(nn.Conv2d(24 + n_classes, width, 3, padding=1, bias=False),
                                     nn.BatchNorm2d(width), nn.ReLU(inplace=True))
        self.correction = nn.Conv2d(width, n_classes, 1)
        nn.init.zeros_(self.correction.weight)
        nn.init.zeros_(self.correction.bias)

    def logits_and_correction(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Stride-8 logits `(B, C, H / 8, W / 8)` and stride-4 correction `(B, C, H / 4, W / 4)`, one backbone pass."""
        feats, h = {}, x
        for name, module in self.base.net.backbone.items():
            h = module(h)
            if name in (STRIDE4_LAYER, LOW_LAYER, HIGH_LAYER):
                feats[name] = h
        logits8 = self.base.net.classifier({"low": feats[LOW_LAYER], "high": feats[HIGH_LAYER]})
        up = F.interpolate(logits8, size=feats[STRIDE4_LAYER].shape[-2:], mode="bilinear", align_corners=False)
        return logits8, self.correction(self.decoder(torch.cat([feats[STRIDE4_LAYER], up], 1)))

    def coarse_logits(self, x: torch.Tensor) -> torch.Tensor:
        return self.logits_and_correction(x)[0]

    def correction_logits(self, x: torch.Tensor) -> torch.Tensor:
        return self.logits_and_correction(x)[1]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        logits8, correction4 = self.logits_and_correction(x)
        size = x.shape[-2:]
        return (F.interpolate(logits8, size=size, mode="bilinear", align_corners=False)
                + F.interpolate(correction4, size=size, mode="bilinear", align_corners=False))

In [ ]:
def build_model(arm: str) -> nn.Module:
    """The original checkpoint (for `original` and `control`) or the checkpoint with the stride-4 decoder added."""
    base = load_segmenter(REPO_ROOT / INIT_CHECKPOINT, "cpu", architecture="lraspp")
    return Stride4LRASPP(base, DECODER_WIDTH) if arm == "stride4" else base

In [ ]:
def n_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

In [ ]:
{arm: n_parameters(build_model(arm)) for arm in ARMS}

In [ ]:
with torch.no_grad():
    probe = torch.randn(2, 3, 256, 256)
    base, wrapped = build_model("original").eval(), build_model("stride4").eval()
    identity_gap = (base(probe) - wrapped(probe)).abs().max().item()
assert identity_gap == 0.0, identity_gap
identity_gap

# Run

In [ ]:
def optimiser(model: nn.Module) -> tuple[torch.optim.Optimizer, torch.optim.lr_scheduler.LRScheduler]:
    """AdamW with the pretrained weights at `LR` and the new decoder at `DECODER_LR`, one-cycle schedule."""
    new = [p for name, p in model.named_parameters() if not name.startswith("base.")] if isinstance(
        model, Stride4LRASPP) else []
    old = [p for p in model.parameters() if all(p is not q for q in new)]
    groups = [{"params": old, "lr": LR}] + ([{"params": new, "lr": DECODER_LR}] if new else [])
    opt = torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[g["lr"] for g in groups], total_steps=STEPS,
                                                pct_start=WARMUP_FRAC)
    return opt, sched

In [ ]:
def train_step(model: nn.Module, opt: torch.optim.Optimizer, sched, rng: np.random.Generator) -> dict:
    """One optimiser step on CE + Dice (= `ce_dice_loss`); returns both terms and their sum."""
    x, y = vs.batch_of(sampler, rng, SAT_BATCH)
    logits, y = model(x.to(device)), y.to(device)
    ce, dice = F.cross_entropy(logits, y, ignore_index=IGNORE_LABEL), dice_loss(logits, y)
    loss = ce + dice
    opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return {"train_ce": ce.item(), "train_dice": dice.item(), "train_loss": loss.item()}

In [ ]:
def log_line(entry: dict) -> str:
    return "  ".join(f"{k}={v:.4f}" if isinstance(v, float) else f"{k}={v}" for k, v in entry.items())

In [ ]:
def train(model: nn.Module, arm: str) -> tuple[nn.Module, list[dict]]:
    """`STEPS` of CE + Dice on the seeded sampler stream (identical for every arm), val metrics every
    `LOG_EVERY_STEPS` (and at step 0), then satellite BN re-estimation and a final val pass."""
    set_seed(SEED)
    rng, model = np.random.default_rng(SEED), model.to(device).train()
    opt, sched = optimiser(model)
    history, window, start = [{"arm": arm, "step": 0, "stage": "train", **val_metrics(model)}], [], time.time()
    bar = tqdm(range(1, STEPS + 1), desc=f"train {arm}", unit="step")
    for step in bar:
        window.append(train_step(model, opt, sched, rng))
        bar.set_postfix(loss=f"{window[-1]['train_loss']:.4f}")
        if step % LOG_EVERY_STEPS == 0 or step == STEPS:
            history.append({"arm": arm, "step": step, "stage": "train", "minutes": (time.time() - start) / 60,
                            "lr": sched.get_last_lr()[0], **pd.DataFrame(window).mean().to_dict(), **val_metrics(model)})
            bar.write(log_line(history[-1])); window = []
    model = vs.domain_bn(model, sampler, SAT_BATCH, BN_BATCHES, BN_SEED, device)
    history.append({"arm": arm, "step": STEPS, "stage": "after BN re-estimation", **val_metrics(model)})
    return model.eval(), history

In [ ]:
@torch.no_grad()
def probabilities(model: nn.Module, rgb: np.ndarray) -> np.ndarray:
    """`(C, H, W)` softmax of x8-upsampled stride-8 logits, plus the x4-upsampled stride-4 correction for a
    `Stride4LRASPP`. Tiled like `lraspp_outputs`, so the base path is exactly the harness's."""
    x, decoder = normalise(rgb), isinstance(model, Stride4LRASPP)
    h, w = x.shape[:2]
    if max(h, w) > SINGLE_PASS_MAX_PX:
        tiled = partial(tiled_logits, x=pad_to_multiple(x, 8), patch=TILE_PX, step=TILE_STEP_PX,
                        n_out=len(CLASSES), dev=EVAL_DEVICE)
        logits8 = tiled(model.coarse_logits, out_stride=8)
        correction4 = tiled(model.correction_logits, out_stride=4) if decoder else None
    else:
        batch = torch.from_numpy(pad_to_multiple(x, ENCODER_STRIDE_PX).transpose(2, 0, 1).copy())[None].to(EVAL_DEVICE)
        outs = model.logits_and_correction(batch) if decoder else (model.coarse_logits(batch), None)
        logits8, correction4 = [o[0].float().cpu().numpy() if o is not None else None for o in outs]
    logits = upsample_logits(logits8, (h, w)) + (upsample_logits(correction4, (h, w), stride=4) if decoder else 0)
    return torch.from_numpy(logits).softmax(0).numpy()

In [ ]:
cases = run_model(build_model("original").to(EVAL_DEVICE).eval(), val_sets, EVAL_DEVICE)

In [ ]:
def val_tiles() -> tuple[np.ndarray, np.ndarray]:
    """Every val image cut into `VAL_CROP_PX` tiles (last row/column overlaps); tiles with no labels dropped."""
    rgbs, labels = [], []
    for case in (c for items in cases.values() for c in items):
        h, w = case.image.label.shape
        for top in tile_starts(h, VAL_CROP_PX):
            for left in tile_starts(w, VAL_CROP_PX):
                win = np.s_[top:top + VAL_CROP_PX, left:left + VAL_CROP_PX]
                if (case.image.label[win] != IGNORE_LABEL).any():
                    rgbs.append(case.image.rgb[win]); labels.append(case.image.label[win])
    return np.stack(rgbs), np.stack(labels)

In [ ]:
val_rgb, val_label = val_tiles()
val_rgb.shape, val_label.shape

In [ ]:
def confusion_miou(confusion: np.ndarray) -> float:
    """mIoU over classes with labelled pixels, from a `(C, C)` label x prediction confusion matrix."""
    inter = np.diag(confusion)
    union = confusion.sum(0) + confusion.sum(1) - inter
    present = confusion.sum(1) > 0
    return float((inter[present] / union[present]).mean())

In [ ]:
@torch.no_grad()
def val_metrics(model: nn.Module) -> dict:
    """CE, Dice loss, their sum (batch means) and pixel mIoU on the val tiles, full-res forward, eval mode."""
    model.eval()
    n, sums, confusion = len(CLASSES), {"val_ce": [], "val_dice": []}, np.zeros((len(CLASSES),) * 2, np.int64)
    for s in range(0, len(val_rgb), VAL_BATCH):
        x = torch.from_numpy(np.stack([normalise(r).transpose(2, 0, 1) for r in val_rgb[s:s + VAL_BATCH]])).to(device)
        y = torch.from_numpy(val_label[s:s + VAL_BATCH].astype(np.int64)).to(device)
        logits, ok = model(x), y != IGNORE_LABEL
        sums["val_ce"].append(F.cross_entropy(logits, y, ignore_index=IGNORE_LABEL).item())
        sums["val_dice"].append(dice_loss(logits, y).item())
        confusion += np.bincount((y[ok] * n + logits.argmax(1)[ok]).cpu().numpy(), minlength=n * n).reshape(n, n)
    model.train()
    out = {k: float(np.mean(v)) for k, v in sums.items()}
    return {**out, "val_loss": out["val_ce"] + out["val_dice"], "val_miou": confusion_miou(confusion)}

In [ ]:
def predict_all(model: nn.Module, arm: str) -> dict[str, list[np.ndarray]]:
    """Class ids for every val image."""
    model = model.to(EVAL_DEVICE).eval()
    return {name: [probabilities(model, c.image.rgb).argmax(0).astype(np.uint8)
                   for c in tqdm(items, desc=f"predict {arm} {name}")] for name, items in cases.items()}

In [ ]:
models, histories, train_minutes = {}, [], {}
for arm in ARMS:
    start = time.time()
    model = build_model(arm)
    if arm == "original":
        histories.append({"arm": arm, "step": 0, "stage": "no training", **val_metrics(model.to(device))})
        print(log_line(histories[-1]))
    else:
        model, history = train(model, arm)
        histories += history
        torch.save(model.state_dict(), REPO_ROOT / OUT_DIR / f"model-{arm}-seed{SEED}.pt")
    models[arm], train_minutes[arm] = model.cpu().eval(), (time.time() - start) / 60
train_minutes

In [ ]:
history_frame = pd.DataFrame(histories)
history_frame.round(4)

In [ ]:
preds = {arm: predict_all(model, arm) for arm, model in models.items()}

In [ ]:
harness_check = {name: float(np.mean([(p == bilinear_argmax(c.output, c.image.rgb)).mean()
                                        for p, c in zip(preds["original"][name], cases[name])])) for name in VAL_SETS}
harness_check

In [ ]:
results = {arm: evaluate_predictions(p, cases) for arm, p in preds.items()}
pd.concat({arm: results_table(r) for arm, r in results.items()}).round(4)

In [ ]:
def components(mask: np.ndarray) -> tuple[np.ndarray, int]:
    """Connected components with fewer than `MIN_BUILDING_PX` pixels removed, relabelled 1..n."""
    lab, n = ndi.label(mask, STRUCTURE)
    keep = np.bincount(lab.ravel(), minlength=n + 1) >= MIN_BUILDING_PX
    keep[0] = False
    relabel = np.zeros(n + 1, np.int32)
    relabel[keep] = np.arange(1, keep.sum() + 1)
    return relabel[lab], int(keep.sum())

In [ ]:
def overlap_fractions(gt: np.ndarray, n_gt: int, pred: np.ndarray, n_pred: int) -> np.ndarray:
    """`(n_gt, n_pred)`: share of each labelled building's area covered by each predicted component."""
    both = (gt > 0) & (pred > 0)
    pairs = np.bincount((gt[both] - 1) * n_pred + (pred[both] - 1), minlength=n_gt * n_pred)
    area = np.bincount(gt.ravel(), minlength=n_gt + 1)[1:]
    return pairs.reshape(n_gt, n_pred) / np.maximum(area, 1)[:, None]

In [ ]:
def building_status(frac: np.ndarray) -> list[str]:
    """Per labelled building: missed, merged (its component also covers another building) or separate."""
    hit = frac >= MATCH_FRAC
    shared = hit.sum(0) >= 2
    return ["missed" if not row.any() else "merged" if shared[row].any() else "separate" for row in hit]

In [ ]:
def status_of(gt: np.ndarray, n_gt: int, mask: np.ndarray) -> list[str]:
    pred, n_pred = components(mask)
    return building_status(overlap_fractions(gt, n_gt, pred, n_pred))

In [ ]:
def building_table(name: str, index: int) -> pd.DataFrame:
    """One row per labelled building of one val image: status under the stride-8 and stride-4 oracles and each arm."""
    label = cases[name][index].image.label
    valid = label != IGNORE_LABEL
    gt, n_gt = components(label == BUILDING)
    table = pd.DataFrame({"set": name, "image": index, "building": np.arange(1, n_gt + 1)})
    for stride in (8, 4):
        table[f"oracle{stride}"] = status_of(gt, n_gt, (label_at_stride(label, "bilinear", stride) == BUILDING) & valid)
    for arm in ARMS:
        table[arm] = status_of(gt, n_gt, (preds[arm][name][index] == BUILDING) & valid)
    return table

In [ ]:
jobs = [(name, i) for name, items in cases.items() for i in range(len(items))]
buildings = pd.concat([building_table(name, i) for name, i in tqdm(jobs, desc="buildings")], ignore_index=True)
buildings

In [ ]:
def merge_summary(frame: pd.DataFrame) -> pd.DataFrame:
    """Per arm: status shares, and the merge rate among buildings each oracle keeps separate."""
    rows = {}
    for arm in ARMS:
        row = frame[arm].value_counts(normalize=True).reindex(STATUSES).fillna(0).to_dict()
        for stride in (8, 4):
            separable = frame[frame[f"oracle{stride}"] == "separate"]
            row[f"merge rate | oracle{stride} separate"] = (separable[arm] == "merged").mean()
        rows[arm] = row
    return pd.DataFrame(rows).T

In [ ]:
summary = pd.concat({name: merge_summary(frame) for name, frame in [("all", buildings), *buildings.groupby("set")]})
summary.round(3)

In [ ]:
mean_over_sets = summary.drop(index="all").groupby(level=1).mean().reindex(ARMS)
mean_over_sets.round(3)

In [ ]:
name, date, top, left = LATENCY_TILE
tile = standard_tile(next(a for a in val_sets[name] if a.date == date).rgb, top, left)
latency = {arm: cpu_latency(lambda m=copy.deepcopy(model): probabilities(m, tile),
                            runs=LATENCY_RUNS, threads=LATENCY_THREADS) for arm, model in models.items()}
pd.DataFrame({arm: vars(l) for arm, l in latency.items()}).T.round(1)

# Plots

In [ ]:
fig_loss, axes = plt.subplots(2, 2, figsize=(13, 7.5))
curves = history_frame[history_frame.stage == "train"]
pairs = [("train_loss", "val_loss", "CE + Dice"), ("train_ce", "val_ce", "CE"), ("train_dice", "val_dice", "Dice loss"),
         (None, "val_miou", "val mIoU (512 tiles)")]
for ax, (train_key, val_key, title) in zip(axes.ravel(), pairs):
    for (arm, frame), colour in zip(curves.groupby("arm"), ["tab:blue", "tab:orange"]):
        if train_key:
            ax.plot(frame["step"], frame[train_key], "-", color=colour, alpha=.5, label=f"{arm} train")
        ax.plot(frame["step"], frame[val_key], "-o", ms=3, color=colour, label=f"{arm} val")
    original = history_frame[history_frame.arm == "original"]
    ax.axhline(original[val_key].item(), color="grey", ls="--", lw=.8, label="original, val")
    ax.set_title(title); ax.set_xlabel("step"); ax.grid(alpha=.3); ax.legend(fontsize=7)
fig_loss.tight_layout()

In [ ]:
fig_metrics, axes = plt.subplots(1, 4, figsize=(16, 3.6))
for ax, key in zip(axes, ["miou", "mdice", "biou", "bf1_2px"]):
    values = [results[arm][MEAN_KEY][key] for arm in ARMS]
    bars = ax.bar(ARMS, values, color=["grey", "tab:blue", "tab:orange"])
    ax.bar_label(bars, fmt="%.4f", fontsize=8); ax.set_title(f"{key} (mean over val sets)"); ax.grid(axis="y", alpha=.3)
fig_metrics.tight_layout()

In [ ]:
fig_merge, axes = plt.subplots(1, 2, figsize=(12, 3.8))
mean_over_sets[STATUSES].plot.bar(stacked=True, ax=axes[0], color=["tab:green", "tab:orange", "tab:red"])
axes[0].set_title("labelled buildings, mean over val sets"); axes[0].tick_params(axis="x", labelrotation=0)
rate_cols = [c for c in mean_over_sets.columns if c.startswith("merge rate")]
mean_over_sets[rate_cols].plot.bar(ax=axes[1]); axes[1].set_title("merge rate among separable buildings")
axes[1].tick_params(axis="x", labelrotation=0); axes[1].legend(fontsize=8)
fig_merge.tight_layout()

In [ ]:
def colour(lab: np.ndarray) -> np.ndarray:
    out = np.full((*lab.shape, 3), 255, np.uint8)
    ok = lab != IGNORE_LABEL
    out[ok] = SEG_PALETTE[lab[ok]]
    return out

In [ ]:
def example_row(axes: np.ndarray, example: list) -> None:
    """RGB | label | each arm's prediction, for one `ZOOM_PX` window."""
    name, index, top, left = example
    case, win = cases[name][index], np.s_[top:top + ZOOM_PX, left:left + ZOOM_PX]
    panels = [(case.image.rgb[win], f"{name} {case.image.date}"), (colour(case.image.label[win]), "label")]
    panels += [(colour(preds[arm][name][index][win]), f"{arm} (stride {ARM_STRIDE[arm]})") for arm in ARMS]
    for ax, (img, title) in zip(axes, panels):
        ax.imshow(img, interpolation="nearest"); ax.set_title(title, fontsize=9); ax.axis("off")

In [ ]:
fig_examples, axes = plt.subplots(len(EXAMPLES), 2 + len(ARMS), figsize=(3.6 * (2 + len(ARMS)), 3.8 * len(EXAMPLES)))
for row, example in zip(np.atleast_2d(axes), EXAMPLES):
    example_row(row, example)
fig_examples.legend([plt.Rectangle((0, 0), 1, 1, color=SEG_PALETTE[c] / 255) for c in range(len(CLASSES))], CLASSES,
                    loc="lower center", ncol=len(CLASSES))
fig_examples.tight_layout(rect=(0, .04, 1, 1))

In [ ]:
for fig, stem in [(fig_loss, "loss"), (fig_metrics, "metrics"), (fig_merge, "merges"), (fig_examples, "examples")]:
    fig.savefig(REPO_ROOT / OUT_DIR / "figures" / f"{stem}-seed{SEED}.png", dpi=120, bbox_inches="tight")

# Log

In [ ]:
record = {"params": {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, list))},
          "history": histories, "train_minutes": train_minutes,
          "harness": {arm: r for arm, r in results.items()},
          "merges": {"summary": summary.reset_index().to_dict("records"), "mean_over_sets": mean_over_sets.to_dict("index")},
          "latency_ms": {arm: vars(l) for arm, l in latency.items()},
          "n_parameters": {arm: n_parameters(m) for arm, m in models.items()}}
(REPO_ROOT / OUT_DIR / f"results-seed{SEED}.json").write_text(json.dumps(record, indent=1, default=float))
buildings.to_csv(REPO_ROOT / OUT_DIR / "runs" / f"buildings-seed{SEED}.csv", index=False)
history_frame.to_csv(REPO_ROOT / OUT_DIR / f"history-seed{SEED}.csv", index=False)
sorted(p.name for p in (REPO_ROOT / OUT_DIR).glob(f"*seed{SEED}*"))